# Notebook 03 : Interprétabilité du Modèle (Explainable AI - SHAP)
**Projet** : Détection de Fraude aux Paiements par Carte Bancaire  
**Objectif** : Expliciter les décisions du modèle XGBoost avec les valeurs SHAP (SHapley Additive exPlanations) pour le rapport de stage et la soutenance devant le jury et la banque.

---

## 1. Importation des Modules & Chargement du Meilleur Modèle (XGBoost)

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.model_selection import train_test_split

sys.path.insert(0, os.path.abspath(".."))
from src.preprocessing import preprocess_data

model_path = "../models/best_model_sota.joblib" if os.path.exists("../models/best_model_sota.joblib") else "models/best_model_sota.joblib"
model = joblib.load(model_path)
print(f"Modèle XGBoost chargé avec succès depuis {model_path} !")

Modèle XGBoost chargé avec succès depuis ../models/best_model_sota.joblib !


## 2. Préparation de l'Échantillon de Validation pour le Calcul SHAP

In [2]:
train_path = "../data/fraudTrain.csv" if os.path.exists("../data/fraudTrain.csv") else "data/fraudTrain.csv"
df_train_full = pd.read_csv(train_path)

X_full, y_full, _ = preprocess_data(df_train_full, is_train=True, use_advanced_features=True)
_, X_val, _, y_val = train_test_split(X_full, y_full, test_size=0.2, stratify=y_full, random_state=42)

# Échantillonnage équilibré de 500 transactions pour le calcul explicatif SHAP
fraud_idx = y_val[y_val == 1].index[:250]
legit_idx = y_val[y_val == 0].index[:250]
sample_idx = fraud_idx.union(legit_idx)

X_sample = X_val.loc[sample_idx]
y_sample = y_val.loc[sample_idx]

print(f"Données préparées. Échantillon SHAP : {len(X_sample)} transactions ({sum(y_sample==0)} Légitimes, {sum(y_sample==1)} Fraudes).")

Données préparées. Échantillon SHAP : 500 transactions (250 Légitimes, 250 Fraudes).


## 3. Calcul des Valeurs SHAP (TreeExplainer)
SHAP attribue à chaque variable son impact exact sur la décision d'alerte.

In [3]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_sample)
print("Calcul SHAP terminé avec succès !")

Calcul SHAP terminé avec succès !


## 4. SHAP Summary Plot (Importance Globale des Variables)
Ce graphique classe les variables par ordre d'importance et montre l'effet des valeurs élevées (rouge) et faibles (bleu) sur le risque de fraude.

In [4]:
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, show=False)
plt.title("Importance Globale des Caractéristiques (SHAP Summary Plot)", fontsize=14, pad=15)
plt.tight_layout()
output_fig1 = "../reports/figures/shap_summary_plot.png" if os.path.exists("../reports") else "reports/figures/shap_summary_plot.png"
plt.savefig(output_fig1, dpi=300, bbox_inches='tight')
plt.show()
print(f"Figure sauvegardée dans : {output_fig1}")

## 5. SHAP Waterfall Plot (Explication Locale d'une Alerte Fraude)
Détail pas-à-pas montrant comment le modèle a évalué une transaction frauduleuse spécifique pour la bloquer.

In [5]:
fraud_idx_in_sample = np.where(y_sample.values == 1)[0][0]

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values[fraud_idx_in_sample], show=False)
plt.title("Explication Pas-à-Pas d'une Alerte Fraude Bloquée (SHAP Waterfall)", fontsize=13, pad=15)
plt.tight_layout()
output_fig2 = "../reports/figures/shap_waterfall_fraud_example.png" if os.path.exists("../reports") else "reports/figures/shap_waterfall_fraud_example.png"
plt.savefig(output_fig2, dpi=300, bbox_inches='tight')
plt.show()
print(f"Figure sauvegardée dans : {output_fig2}")

## 6. Synthèse d'Interprétabilité pour le Rapport Final
1. **`velocity_kmh` (Vitesse de déplacement)** : Facteur n°1 de détection. Une vitesse > 100 km/h entre deux achats consécutifs fait grimper le score de risque de manière critique.
2. **`amt_mad` / `amt_diff_from_card_avg` (Montant en Dirhams)** : Les achats d'un montant élevé et s'écartant fortement de la moyenne du client déclenchent l'alerte.
3. **`dist_home_to_merch_km` (Distance géographique)** : Les achats effectués très loin du domicile habituel du client augmentent la probabilité de fraude.